In [ ]:
from pathlib import Path
import cv2
from rtmlib import PoseTracker, Wholebody, draw_skeleton
import csv
import numpy as np
from tqdm.auto import tqdm



VIDEO_PATH = Path("__file__").parent / "data" / "bench_ludo.mov"
IMG_PATH = Path("__file__").parent / "data" / "test.jpeg"

OUTPUT_DIR = Path("__file__").parent / "outputs"
CONFIDENCE_THRESHOLD = 0.4


LEFT_HIP = 11
RIGHT_HIP = 12
LEFT_KNEE = 13
RIGHT_KNEE = 14
LEFT_ANKLE = 15
RIGHT_ANKLE = 16



/Users/yanicrothlingshofer/micromamba/envs/gym-nerd/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
image = cv2.imread(str(IMG_PATH))


model = Wholebody(
    mode="performance",
    backend="onnxruntime",
    device="cpu",
    to_openpose=False,
)

keypoints, scores = model(image)

result = draw_skeleton(
    image.copy(),
    keypoints,
    scores,
    openpose_skeleton=False,
    kpt_thr=0.4,
    
)

cv2.imwrite("resultat.jpg", result)

print("Forme des keypoints :", keypoints.shape)
print("Forme des scores :", scores.shape)

load /Users/yanicrothlingshofer/.cache/rtmlib/hub/checkpoints/yolox_m_8xb8-300e_humanart-c2c7a14a.onnx with onnxruntime backend


2026-08-30 15:32:55.681 python[62393:29190411] 2026-08-30 15:32:55.680882 [W:onnxruntime:, graph.cc:5583 CleanUnusedInitializersAndNodeArgs] Removing initializer '1709'. It is not used by any node and should be removed from the model.
2026-08-30 15:32:55.681 python[62393:29190411] 2026-08-30 15:32:55.681003 [W:onnxruntime:, graph.cc:5583 CleanUnusedInitializersAndNodeArgs] Removing initializer '1701'. It is not used by any node and should be removed from the model.
2026-08-30 15:32:55.681 python[62393:29190411] 2026-08-30 15:32:55.681035 [W:onnxruntime:, graph.cc:5583 CleanUnusedInitializersAndNodeArgs] Removing initializer '1706'. It is not used by any node and should be removed from the model.


load /Users/yanicrothlingshofer/.cache/rtmlib/hub/checkpoints/rtmw-dw-x-l_simcc-cocktail14_270e-384x288_20231122.onnx with onnxruntime backend
Forme des keypoints : (1, 133, 2)
Forme des scores : (1, 133)


In [4]:
video_output_path = OUTPUT_DIR / "output_video.mp4"

cap = cv2.VideoCapture(str(VIDEO_PATH))

if not cap.isOpened():
    print("Erreur lors de l'ouverture de la vidéo.")
    exit()

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

if fps == 0:
    print("Erreur : FPS de la vidéo est 0.")
    exit()

writer = cv2.VideoWriter(
    str(video_output_path),
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height) 
)

model = Wholebody(
    mode="performance",
    backend="onnxruntime",
    device="cpu",
    to_openpose=False,
)

try:
    for fram_idx in tqdm(range(frame_count), desc="Processing frames"):
        ret, frame = cap.read()
        if not ret:
            print(f"Erreur lors de la lecture de la frame {fram_idx}.")
            break

        keypoints, scores = model(frame)

        result_frame = draw_skeleton(
            frame.copy(),
            keypoints,
            scores,
            openpose_skeleton=False,
            kpt_thr=CONFIDENCE_THRESHOLD,
        )

        writer.write(result_frame)
finally:
    cap.release()
    writer.release()
    cv2.destroyAllWindows()

print(f"Vidéo traitée et enregistrée dans : {video_output_path}")

load /Users/yanicrothlingshofer/.cache/rtmlib/hub/checkpoints/yolox_m_8xb8-300e_humanart-c2c7a14a.onnx with onnxruntime backend


2026-08-30 15:32:59.608 python[62393:29190411] 2026-08-30 15:32:59.607554 [W:onnxruntime:, graph.cc:5583 CleanUnusedInitializersAndNodeArgs] Removing initializer '1709'. It is not used by any node and should be removed from the model.
2026-08-30 15:32:59.608 python[62393:29190411] 2026-08-30 15:32:59.608073 [W:onnxruntime:, graph.cc:5583 CleanUnusedInitializersAndNodeArgs] Removing initializer '1701'. It is not used by any node and should be removed from the model.
2026-08-30 15:32:59.608 python[62393:29190411] 2026-08-30 15:32:59.608099 [W:onnxruntime:, graph.cc:5583 CleanUnusedInitializersAndNodeArgs] Removing initializer '1706'. It is not used by any node and should be removed from the model.


load /Users/yanicrothlingshofer/.cache/rtmlib/hub/checkpoints/rtmw-dw-x-l_simcc-cocktail14_270e-384x288_20231122.onnx with onnxruntime backend


Processing frames: 100%|██████████| 261/261 [02:08<00:00,  2.03it/s]

Vidéo traitée et enregistrée dans : output_video.mp4
